# 📗 하이브리드 검색

## 하이브리드 검색은 무엇인가요?

**하이브리드 검색(Hybrid Search)은 서로 다른 방식으로 찾은 문서들을 결합해 하나의 최종 검색 결과로 만드는 방법입니다.** 이 교안에서는 같은 문서 집합에 **BM25 어휘 검색**과 **Dense 임베딩 검색**을 적용합니다.

<img src="images/01_hybrid_search.png" width="1100" alt="같은 질문과 문서 집합을 BM25와 Dense로 각각 검색한 뒤, 원문 ID별로 RRF 점수를 합산하고 최종 상위 문서를 선택하는 흐름">

같은 질문으로 같은 문서 집합을 두 방식으로 검색합니다.

| 검색 방식 | 비교 기준 |
|---|---|
| BM25 | 같은 토큰의 등장 횟수·희귀도와 문서 길이 |
| Dense | 질문 벡터와 문서 벡터 사이의 거리 |

**어휘 검색이 필요한 이유:** Dense는 의미가 비슷해도 질문에 쓴 단어가 없는 문서를 반환할 수 있습니다. 제품명·오류 코드·전문용어처럼 정확한 표현이 중요한 검색에서는, 전처리 후 같은 토큰의 일치를 점수에 반영하는 BM25가 이를 보완합니다.

**두 방식을 결합해 한쪽이 놓친 근거를 다른 쪽에서 찾을 기회를 늘립니다.**

1. BM25와 Dense가 각각 상위 3개를 찾습니다.
2. 가중 RRF(Reciprocal Rank Fusion, 역순위 융합)로 등수를 점수로 바꾸고, 같은 원문 ID의 점수를 합산해 문서는 한 번만 남깁니다.
3. 합쳐진 최대 6개 후보 중 점수가 높은 최종 3개를 선택합니다.

검색 품질은 같은 질문·문서 집합·최종 개수에서 필요한 근거를 얼마나 찾았는지로 비교합니다.

## Advanced RAG에서 이번 교안의 위치를 살펴봅니다

기본 RAG는 검색한 문서를 LLM에 전달해 답변을 생성합니다. **Advanced RAG는 근거 누락과 불필요한 내용의 유입을 줄이도록 검색 전·검색·검색 후 단계를 개선합니다.**

| 단계 | 하는 일 | 학습 순서 |
|---|---|---|
| 검색 전(Pre-retrieval) | 문서 분할·인덱스 구성·질의 변환 | 지난 시간: 청킹·RAPTOR / 이번 교안 02: 질의 변환 |
| 검색(Retrieval) | 후보 검색과 검색 결과 결합 | **이번 교안 01: 하이브리드 검색** |
| 검색 후(Post-retrieval) | 후보 재정렬·질문에 필요한 내용 추출 | 다음 시간: 리랭킹·컨텍스트 압축 |

## 오늘의 목표

- [ ] BM25가 희귀도·빈도·문서 길이로 점수를 매기는 방식을 수식으로 설명할 수 있습니다.
- [ ] 같은 질문으로 BM25와 Dense 검색 결과를 비교할 수 있습니다.
- [ ] RRF로 두 순위를 합치고 `EnsembleRetriever`로 하이브리드 검색기를 만들 수 있습니다.
- [ ] 같은 상위 3개에서 Recall과 결과 글자 수를 비교할 수 있습니다.

## ⏪ 지난 시간 복습

앞 단원에서는 Advanced RAG 단계표의 **검색 전** 단계를 청킹 전략과 RAPTOR로 바꿨습니다. 교안 01은 같은 표의 **검색** 단계입니다. 임베딩 검색(Dense)에 단어 일치 검색(BM25)을 더해 한쪽이 놓친 문서를 다른 쪽이 채우게 합니다. 이런 보완 관계를 **상보성**이라고 합니다.

시연은 가상 도서 목록(제목과 소개문), 따라하기는 앞 단원과 같은 매뉴얼 44절입니다.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하고 준비 셀을 위에서부터 실행하세요. 경로(`material_dir`·`data_dir`·`output_dir`)와 `read_json`·`save_json`은 앞 단원과 같습니다. 첫 셀에서 `langchain-community` 유지보수 종료를 알리는 경고가 한 번 보일 수 있습니다. `BM25Retriever`를 이 패키지에서 가져오기 때문이며 실행에는 문제가 없습니다.


In [ ]:
# 문서·검색기·모델에 필요한 라이브러리를 가져옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings


In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


임베딩은 `text-embedding-3-large`의 768차원입니다. `check_embedding_ctx_length=False`는 자동 길이 검사·분할을 끕니다. API 키는 `.env`에서 읽습니다.


In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 적재·검색 셀에서 요청합니다.")


절 하나가 이미 짧은 발췌라서 다시 나누지 않고 레코드 하나를 검색 단위 하나로 씁니다. 이 단위를 고정한 채 검색 방식과 질문을 바꿔 결과를 비교합니다. `make_documents`는 원문 ID·제목·출처에 필터용 메타데이터를 더해 `Document`를 만듭니다.


In [ ]:
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 사용하고 출처·필터 메타데이터를 담습니다.
    return [Document(
        id=record["doc_id"],
        page_content=record["text"],
        metadata={"source_id": record["doc_id"], "title": record["title"],
                  "url": record["url"], "source": record["source"], **record["metadata"]},
    ) for record in records]


In [ ]:
# 전체 목록을 먼저 보고 질문에 필요한 필터 필드를 확인합니다.
records = read_json("demo_docs.json")
display(pd.DataFrame(records)[["doc_id", "title", "metadata"]])
print(records[0]["text"])
documents = make_documents(records)


In [ ]:
def show_results(documents):
    """앞 5개 결과의 원문 ID·메타데이터·본문을 모든 열과 함께 보여 줍니다."""
    # 빈 결과는 필터를 바꾸지 않고 그대로 알립니다.
    if not documents:
        print("조건에 맞는 검색 결과가 없습니다.")
        return
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    # 열 순서를 고정해야 표를 나란히 비교할 수 있습니다. url은 길어서 뺍니다.
    rows = [{"display_order": order, "source_id": doc.metadata["source_id"], "title": doc.metadata["title"],
             **{key: doc.metadata[key] for key in sorted(doc.metadata) if key not in {"source_id", "title", "url"}},
             "text": doc.page_content} for order, doc in enumerate(documents, start=1)]
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head())

따라하기 원문은 앞 단원과 같은 44개 절입니다. 원본은 `data/originals/`에 있고, `pdf_page`는 PDF 뷰어에서 1부터 세는 페이지, `category`는 수업용 분류입니다.


In [ ]:
# 따라하기는 도서 목록과 다른 도메인의 실제 매뉴얼을 사용합니다.
practice_records = read_json("practice_docs.json")
practice_documents = make_documents(practice_records)
show_results(practice_documents)


## 1. 한국어 토큰으로 BM25 검색하기

BM25는 질문과 문서를 토큰으로 나눈 뒤, **같은 토큰이 문서에 등장하는지와 몇 번 등장하는지** 등을 이용해 점수를 계산하는 검색 방법입니다. 예를 들어 질문의 토큰이 `미적분`이면 문서 본문에서도 `미적분`이라는 토큰을 찾습니다. 문서가 같은 주제를 설명하더라도 `미적분` 토큰이 없다면, 뜻이 비슷하다는 이유만으로 그 토큰의 점수를 주지는 않습니다.

한국어는 조사가 붙어 공백으로 나누면 `미적분을`과 `미적분`이 다른 토큰이 됩니다. 그래서 `Kiwi`로 명사·외국어·숫자만 남기는 `kiwi_tokenize`를 만들고, `preprocess_func`로 넘겨 **문서와 질문에 같은 함수**를 씁니다.

`text.replace("･", "·")`는 필수 전처리는 아닙니다. 이 PDF의 `재택･원격근무`를 `재택·원격근무`로 바꾸어 기호 표기를 통일합니다. `･`가 없는 자료에서는 생략해도 됩니다.


In [ ]:
# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF 표기 통일: 재택･원격근무 → 재택·원격근무
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    return [token.form.lower() for token in kiwi.tokenize(text)
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]

In [ ]:
# 조사가 붙은 질문에서 공백 분리와 형태소 토큰의 차이를 읽습니다.
print("공백 분리:", "미적분을 배우는 책".split())
print("형태소 분리:", kiwi_tokenize("미적분을 배우는 책"))


### BM25가 점수를 매기는 기준

BM25는 **질문과 문서에서 문자열이 같은 토큰의 점수를 더해** 문서 점수를 구합니다.

- **DF(등장 문서 수)**: 토큰이 한 번 이상 나오는 문서 수입니다. 전체 문서 수와 함께 희귀도 가중치(IDF)를 계산합니다.
- **TF(문서 안의 등장 횟수)**: 해당 문서에서 토큰이 나온 횟수입니다. 한 문서에 10번 나오면 TF는 10, DF에는 문서 1개로 셉니다.
- **문서 길이**: 전처리 뒤 토큰 수(중복 포함)를 전체 문서의 평균 토큰 수와 비교해 보정합니다.

문서에 없는 질문 토큰의 점수 기여는 0입니다.


<img src="images/04_bm25_principles.png" width="1100" alt="DF는 토큰을 포함한 문서 수, TF는 한 문서 안의 반복 횟수입니다. 평균 문서 길이는 검색 대상 전체의 토큰 수 합계 ÷ 문서 수로 계산하며, 문서 집합마다 달라집니다. 가운데의 100토큰은 반복 횟수를 비교하기 위한 예시입니다. 오른쪽은 같은 반복 횟수에서 문서 길이가 해당 집합의 평균과 같을 때(비율 1), 평균의 두 배일 때(비율 2)를 비교합니다.">

DF는 토큰을 포함한 문서 수, TF는 한 문서 안의 반복 횟수입니다. 평균 문서 길이는 검색 대상 전체의 토큰 수 합계 ÷ 문서 수로 계산하며, 문서 집합마다 달라집니다. 가운데의 100토큰은 반복 횟수를 비교하기 위한 예시입니다. 오른쪽은 같은 반복 횟수에서 문서 길이가 해당 집합의 평균과 같을 때(비율 1), 평균의 두 배일 때(비율 2)를 비교합니다.


### 수식을 의미로 읽기

**문서 점수 = 질문 토큰별 [희귀도 × 반복 횟수·길이 보정값]의 합**

$$
\operatorname{BM25}(Q,d)
= \sum_{t\in Q}
\operatorname{IDF}(t)
\cdot
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1\left(1-b+b\frac{|d|}{\operatorname{avgdl}}\right)}
$$

| 수식 | 의미 |
|---|---|
| $\sum_{t\in Q}$ | 질문의 각 토큰에 대한 점수를 모두 더함 |
| $\operatorname{IDF}(t)$ | 토큰의 희귀도 가중치. 전체 문서 수와 해당 토큰을 포함한 문서 수로 계산 |
| $f(t,d)$ | 이 문서에서 해당 토큰이 등장한 횟수(TF) |
| $\lvert d\rvert/\operatorname{avgdl}$ | 이 문서의 토큰 수 ÷ 전체 문서의 평균 토큰 수 |
| 식의 분수 전체 | 반복 횟수와 문서 길이를 함께 반영한 값 |
| $k_1$ | 클수록 더 많은 반복까지 반영. 같은 길이에서 반복에 따른 증가량은 점차 감소 |
| $b$ | 길이 보정 정도. 0이면 길이 보정을 끄고, 1이면 길이 비율을 그대로 반영 |

**핵심:** IDF가 양수일 때, 같은 길이에서는 반복이 많을수록 점수가 커지되 증가량은 줄어듭니다. 같은 반복 횟수에서는 긴 문서의 점수가 작아집니다($k_1>0$, $b>0$). 여기서 점수는 해당 토큰의 기여를 뜻합니다.

문서에 없는 질문 토큰의 기여는 0입니다. BM25 점수는 정답 확률이 아닙니다.


<details><summary>라이브러리가 IDF를 계산하는 방법</summary>

이 실습의 `BM25Retriever`는 `rank_bm25.BM25Okapi`로 점수를 계산합니다. 전체 문서 수를 $N$, 토큰 $t$가 한 번 이상 등장하는 문서 수(DF)를 $n(t)$라 하면, **보정 전 IDF**는 다음과 같습니다.

$$
\operatorname{IDF}_{\mathrm{raw}}(t)=\ln\left(\frac{N-n(t)+0.5}{n(t)+0.5}\right)
$$

분자는 토큰이 **없는** 문서 수에 0.5를 더한 값이고, 분모는 **있는** 문서 수에 0.5를 더한 값입니다. $N$이 같을 때 $n(t)$가 작을수록 이 비율과 로그값이 커집니다. 0.5를 더하면 $n(t)=0$이나 $N$에서도 분모 또는 로그의 입력이 0이 되지 않습니다.

토큰이 전체 문서의 절반에 있으면 보정 전 IDF는 0이고, 절반보다 많은 문서에 있으면 음수입니다. 예를 들어 문서 44개 중 36개에 등장하면 $\ln(8.5/36.5)\approx-1.46$입니다.

`rank-bm25` 0.2.2는 이 값이 음수인 토큰에만 **epsilon × average_idf**를 대신 사용합니다. 기본 `epsilon`은 0.25이고, `average_idf`는 **문서 집합의 서로 다른 토큰 각각에 대해 구한 보정 전 IDF의 산술평균**입니다. 같은 토큰이 여러 번 나와도 이 평균에는 한 번만 포함합니다. 평균 자체가 음수이면 대체값도 음수일 수 있으므로, 이 보정이 양수를 보장하는 것은 아닙니다.

본문의 ‘TF가 클수록 기여가 커진다’와 ‘길수록 기여가 작아진다’는 설명은 **실제 적용되는 IDF가 양수인 토큰**에 대한 것입니다. 문서 집합에 없는 질문 토큰에는 저장된 IDF가 없으며, 이 구현은 그 토큰의 기여를 0으로 처리합니다.

[BM25Okapi 구현](https://github.com/dorianbrown/rank_bm25/blob/master/rank_bm25.py) · [LangChain BM25Retriever 구현](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/retrievers/bm25.py)

</details>


### BM25 검색기 만들기

`bm25_params`에는 점수 계산 설정을 넣습니다. 여기서는 기본값인 `k1=1.5`, `b=0.75`를 사용합니다. `k=3`은 반환할 최대 문서 수입니다.

`preprocess_func=kiwi_tokenize`는 검색기에서 사용할 토큰화 함수를 지정합니다.

- `bm25.invoke("미적분")`: 질문을 자동으로 토큰화하고, 점수가 높은 상위 문서를 반환합니다.
- `bm25.vectorizer.get_scores(kiwi_tokenize("미적분"))`: 내부 점수 계산기를 직접 호출하므로 질문을 직접 토큰화합니다. 전체 문서의 점수를 입력 문서 순서로 반환합니다.


In [ ]:
bm25 = BM25Retriever.from_documents(
    documents, preprocess_func=kiwi_tokenize,
    bm25_params={"k1": 1.5, "b": 0.75}, k=3,
)

# get_scores는 전처리를 하지 않으므로 질문을 직접 토큰화합니다.
bm25_scores = bm25.vectorizer.get_scores(kiwi_tokenize("미적분"))
print("문서 목록 순서의 BM25 점수:", [round(float(score), 2) for score in bm25_scores])


In [ ]:
# invoke는 저장된 kiwi_tokenize로 질문을 자동 토큰화하므로 문자열을 그대로 전달합니다.
# 점수 내림차순의 상위 문서 3개를 반환합니다.
bm25_results = bm25.invoke("미적분")
show_results(bm25_results)


위 점수는 문서 목록과 같은 순서입니다. ‘미적분’이 든 책 한 권만 점수를 받고 나머지는 0이라, 결과의 2·3번째는 점수가 0인 책입니다. 이 검색기는 최소 점수로 문서를 제외하지 않고 점수순으로 최대 `k`개를 반환하므로 **결과가 있다고 근거가 있는 것은 아닙니다**.


### 🖐️ 함께 따라하기: 매뉴얼에서 정확한 용어 찾기

`practice_documents`로 **practice_bm25**(`kiwi_tokenize`, `bm25_params={'k1': 1.5, 'b': 0.75}`, `k=3`)를 만들고, ‘시차출퇴근 취업규칙’의 검색 결과를 **practice_bm25_results**에 담아 출력하세요.

**확인 기준**: 첫 결과가 `hr_stagger_rules`(시차출퇴근 취업규칙의 작성·신고)입니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 2. 같은 질문을 Dense로 검색하기

Dense 검색은 질문과 문서를 같은 임베딩 모델로 벡터로 바꾸고, 벡터 저장소에 설정된 거리 기준으로 질문 벡터에 가까운 문서를 찾습니다. 문서 벡터는 적재할 때 저장하고, 질문 벡터는 검색할 때 만듭니다. 토큰 문자열이 같지 않아도 벡터가 가까우면 검색될 수 있지만, 답변에 필요한 문서가 반드시 상위에 오는 것은 아닙니다. 이 비교에서는 문서 집합·질문·반환 개수(`k=3`)를 BM25와 같게 둡니다.


In [ ]:
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day47_lesson01_books", embedding_function=embedding_model)
vector_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = vector_store.add_documents(documents)
print("day47_lesson01_books 적재 수:", len(added_ids))


In [ ]:
# 검색 단위와 반환 개수는 BM25와 동일하게 맞춥니다.
dense = vector_store.as_retriever(search_kwargs={"k": 3})
query = "컴퓨터가 예시에서 규칙을 배우는 방법"
dense_results = dense.invoke(query)
print("Dense")
show_results(dense_results)
# 질문 토큰이 어느 문서에도 없으면 모든 문서가 0점이고, 그래도 k개는 채워집니다.
bm25_scores = bm25.vectorizer.get_scores(kiwi_tokenize(query))
print("BM25 점수:", [round(float(score), 2) for score in bm25_scores])
show_results(bm25.invoke(query))

질문의 토큰(컴퓨터·예시·규칙·방법)이 열 권의 소개문 어디에도 없어 BM25 점수는 모두 0입니다. 그래도 검색기는 동점 문서 중 3권을 반환합니다. 이 순서는 관련성의 차이를 나타내지 않으며, 동점 정렬의 세부 동작에 따라 달라질 수 있습니다. Dense 결과에는 기계 학습을 다루는 딥러닝 책이 포함됐는지 원문 ID와 본문으로 확인하세요.


In [ ]:
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
practice_store = Chroma(collection_name="day47_lesson01_hr", embedding_function=embedding_model)
practice_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = practice_store.add_documents(practice_documents)
print("day47_lesson01_hr 적재 수:", len(added_ids))


### 🖐️ 함께 따라하기: 표현을 바꾼 질문 비교하기

`practice_store`로 **practice_dense**(`k=3`)를 만들고, ‘집에서 일할 때 회사 자료를 안전하게 다루려면?’을 `practice_dense`와 1번 따라하기의 `practice_bm25`로 각각 검색해 비교하세요.

**확인 기준**: BM25 결과 3건에는 보안 절 `hr_security`가 없습니다(원문이 ‘자료·안전’ 대신 ‘보안’이라고 씁니다). Dense 결과에 있는지 보세요.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 3. RRF로 두 순위 합치기

BM25 점수와 Dense 검색의 거리·유사도 값은 계산 방식과 범위가 다릅니다. 그대로 더하면 두 검색기의 관련성 판단을 동등하게 반영하지 못하고 숫자의 크기가 큰 쪽에 합산 결과가 좌우될 수 있습니다. RRF(Reciprocal Rank Fusion, 역순위 융합)는 원래 점수를 쓰지 않고 **검색 결과의 등수로 새 점수를 계산**합니다. 여기서는 검색기별 가중치를 적용한 RRF를 사용합니다.

**한 검색기가 문서에 주는 점수 = 검색기 가중치 ÷ (상수 + 그 문서의 등수)**

같은 문서가 여러 검색 결과에 있으면 이 값들을 더합니다.

$$
\operatorname{RRF}(d)
=\sum_{j:\,d\in L_j}\frac{w_j}{c+r_j(d)}
$$

| 기호 | 의미 |
|---|---|
| $L_j$ | $j$번째 검색기가 돌려준 원문 ID 순위 목록 |
| $r_j(d)$ | 그 목록에서 문서 $d$의 등수(1부터) |
| $w_j$ | 그 검색기의 가중치 |
| $c$ | 등수에 더하는 상수(여기서는 60). 같은 양의 가중치에서 값을 크게 할수록 1위와 2위가 받는 점수의 차이가 작아짐. 검색 개수 `k`와는 별개 |

같은 양의 가중치에서는 1위가 2위보다 분모가 작으므로 더 큰 점수를 받습니다. 문서가 어떤 검색기의 반환 목록에 없으면 그 검색기에서 받는 점수는 0입니다. 합산한 RRF 점수가 큰 문서부터 정렬합니다.

**계산 예**: BM25 순위 `B, A, C`, Dense 순위 `D, A, E`, 가중치 0.5씩이면 A는 $\frac{0.5}{62}+\frac{0.5}{62}\approx0.01613$, B는 $\frac{0.5}{61}\approx0.00820$입니다. 두 목록의 2위 A가 한 목록의 1위 B보다 앞섭니다.

원논문은 이 상수를 `k`라 부르지만 `EnsembleRetriever`의 인자 이름을 따라 `c`로 씁니다. [RRF 원논문](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf)

<img src="images/05_rrf_contributions.png" width="1100" alt="A는 두 목록의 기여를 더하고, B·D는 한 목록의 기여만 받습니다.">

A는 두 목록의 기여를 더하고, B·D는 한 목록의 기여만 받습니다.


## 4. EnsembleRetriever로 하이브리드 검색기 만들기

`EnsembleRetriever`는 같은 질문을 각 검색기에 보내고, 반환된 문서마다 가중 RRF 점수를 계산합니다. 두 검색기에 함께 나온 문서는 각 목록에서 받은 점수를 합산하고, 중복을 제거한 뒤 합산 점수가 큰 순서로 반환합니다.

이 실습은 의미 검색에 더 큰 비중을 주어 **BM25 0.3·Dense 0.7**로 설정합니다. 최적의 비율은 문서와 질문에 따라 달라지므로 검색 결과를 비교해 조정합니다.

- `id_key="source_id"`: 같은 문서인지 원문 ID로 판단합니다. 지정하지 않으면 본문 문자열이 같을 때 한 문서로 합칩니다.
- 각 검색기가 서로 다른 문서 3개씩을 반환하면, 두 목록이 모두 같을 때 3개, 전혀 겹치지 않을 때 6개입니다. `hybrid.invoke(query)`는 중복을 제거하고 RRF 점수순으로 정렬한 전체 문서 목록을 반환합니다.


<img src="images/01_hybrid_search.png" width="1100" alt="같은 질문으로 얻은 두 순위를 원문 ID를 기준으로 RRF로 합칩니다.">

같은 질문으로 얻은 두 순위를 원문 ID를 기준으로 RRF로 합칩니다.


In [ ]:
# weights의 순서는 retrievers의 순서와 같습니다.
hybrid = EnsembleRetriever(
    retrievers=[bm25, dense], weights=[0.3, 0.7], c=60, id_key="source_id",
)
hybrid_results = hybrid.invoke(query)
show_results(hybrid_results)


BM25 점수가 0인 문서도 반환 목록에 있으면 RRF에서 `가중치 / (c + 등수)`만큼의 양의 점수를 받습니다. RRF는 원래 BM25 점수가 0이었는지 확인하지 않습니다. 이 문서가 최종 상위 결과에 포함되면 필요한 근거 문서가 제외될 수 있으므로, 하이브리드 검색의 품질은 실제 질문별 결과로 비교해야 합니다.


### 🖐️ 함께 따라하기: 매뉴얼 하이브리드 검색기 만들기

1·2번 따라하기의 `practice_bm25`, `practice_dense`를 가중치 `[0.3, 0.7]`, `c=60`, `id_key='source_id'`로 묶어 **practice_hybrid**를 만들고 ‘재택근무 장비 비용’의 상위 3개를 출력하세요.

**확인 기준**: 결과 3개의 `source_id`가 모두 다릅니다.


In [ ]:
# 위 요구사항에 맞게 코드를 작성하세요.


## 5. Recall@3으로 검색 방식과 가중치 비교하기

**Recall@3 = 상위 3개에서 찾은 정답 문서 수 ÷ 전체 정답 문서 수**

검색기별 후보 5개를 모아 **BM25·Dense·Hybrid 두 가지 가중치**의 최종 3개를 비교합니다. 가중치 순서는 **BM25, Dense**이며, 두 Hybrid는 같은 후보 목록을 사용합니다.

정확한 용어, 다른 표현, 두 가지·세 가지 요구를 묻습니다. 정답이 3개이고 그중 2개를 찾으면 Recall@3은 `2/3`입니다.

상세 결과, 질문별 점수표, **종합 점수(`mean_recall_at_3`)** 순서로 확인합니다. 종합 점수는 네 질문의 Recall@3을 같은 비중으로 평균한 값입니다. 상세 표의 `retrieved_ids`와 `missing_ids`로 찾은 문서와 놓친 정답을 확인하세요. BM25 점수가 모두 0이면 반환 순서는 동점 처리 결과입니다.


In [ ]:
def source_recall(documents, expected_ids):
    """정답 문서 ID 중 검색된 ID의 비율을 반환합니다."""
    expected = set(expected_ids)
    retrieved_ids = {doc.metadata["source_id"] for doc in documents}
    # 분모는 검색 결과 수가 아니라 필요한 원문 수입니다. 빈 집합이면 0으로 나누므로 근거가 있는 질문에만 씁니다.
    return len(retrieved_ids & expected) / len(expected)


In [ ]:
comparison_bm25 = BM25Retriever.from_documents(
    documents, preprocess_func=kiwi_tokenize,
    bm25_params={"k1": 1.5, "b": 0.75}, k=5,
)
comparison_dense = vector_store.as_retriever(search_kwargs={"k": 5})
comparison_hybrids = {
    name: EnsembleRetriever(
        retrievers=[comparison_bm25, comparison_dense],
        weights=weights, c=60, id_key="source_id",
    )
    for name, weights in [("Hybrid 0.5/0.5", [0.5, 0.5]), ("Hybrid 0.3/0.7", [0.3, 0.7])]
}


In [ ]:
comparison_cases = [
    ("정확한 용어", "미적분", {"book04"}),
    ("어휘가 다른 질문", "항성의 일생을 알고 싶어요.", {"book05"}),
    ("두 가지 요구", "미적분과 항성의 일생을 각각 공부하려고 합니다.", {"book04", "book05"}),
    ("세 가지 요구", "미적분, SQL 조인, 항성의 일생을 각각 공부하려고 합니다.",
     {"book02", "book04", "book05"}),
]

# 각 질문을 한 번씩 검색하고 같은 후보로 네 방식을 비교합니다.
comparison_rows = []
for case, comparison_query, expected_ids in comparison_cases:
    candidates = [comparison_bm25.invoke(comparison_query),
                  comparison_dense.invoke(comparison_query)]

    # 같은 후보의 RRF 점수를 가중치별로 계산하고 내림차순으로 정렬합니다.
    method_results = {"BM25": candidates[0], "Dense": candidates[1]}
    for name, retriever in comparison_hybrids.items():
        method_results[name] = retriever.weighted_reciprocal_rank(candidates)

    # 점수가 모두 0이면 순위가 토큰 일치를 구별하지 못한 것입니다.
    bm25_scores = comparison_bm25.vectorizer.get_scores(kiwi_tokenize(comparison_query))
    if all(score == 0 for score in bm25_scores):
        print(f"{case}: BM25 점수가 모두 0입니다. 반환 순서는 동점 처리 결과입니다.")

    # 네 방식 모두 최종 상위 3개에서 정답 문서를 셉니다.
    for method, candidate_documents in method_results.items():
        results = candidate_documents[:3]
        retrieved_ids = [doc.metadata["source_id"] for doc in results]
        comparison_rows.append({
            "case": case, "method": method,
            "recall_at_3": source_recall(results, expected_ids),
            "found_count": len(expected_ids & set(retrieved_ids)),
            "expected_count": len(expected_ids),
            "retrieved_ids": retrieved_ids,
            "missing_ids": sorted(expected_ids - set(retrieved_ids)),
        })

# 상세 결과를 먼저 확인합니다.
comparison_frame = pd.DataFrame(comparison_rows)
with pd.option_context("display.max_columns", None):
    display(comparison_frame.round(3))


In [ ]:
# 질문별 점수를 나란히 봅니다.
score_table = comparison_frame.pivot(index="case", columns="method", values="recall_at_3")
score_table = score_table.reindex(
    index=[case for case, query, expected in comparison_cases],
    columns=["BM25", "Dense", *comparison_hybrids],
)
display(score_table.round(3))


In [ ]:
# 네 질문의 Recall@3을 같은 비중으로 평균합니다.
display(comparison_frame.groupby("method", sort=False)[["recall_at_3"]].mean()
        .rename(columns={"recall_at_3": "mean_recall_at_3"}).round(3))


### 🖐️ 함께 따라하기: 매뉴얼 질문별 검색 결과와 가중치 비교하기

아래 코드를 실행해 **같은 6개 질문**의 BM25·Dense·Hybrid 결과를 비교하세요. Hybrid는 가중치 `0.5·0.5`와 `0.3·0.7`을 비교합니다.

정답은 `relevant_doc_ids`이며, **질문별 Recall@3과 평균**을 함께 확인합니다.


In [ ]:
# [완성 코드]
wanted_ids = {"q1", "q4", "q6", "q11", "q31", "q32"}
practice_questions = [item for item in read_json("practice_questions.json")
                      if item["question_id"] in wanted_ids]

# 검색 후보는 각 5개로 맞추고 두 가중치를 준비합니다.
practice_eval_bm25 = BM25Retriever.from_documents(
    practice_documents, preprocess_func=kiwi_tokenize,
    bm25_params={"k1": 1.5, "b": 0.75}, k=5,
)
practice_eval_dense = practice_store.as_retriever(search_kwargs={"k": 5})
practice_hybrids = {
    name: EnsembleRetriever(
        retrievers=[practice_eval_bm25, practice_eval_dense],
        weights=weights, c=60, id_key="source_id",
    )
    for name, weights in [("Hybrid 0.5/0.5", [0.5, 0.5]), ("Hybrid 0.3/0.7", [0.3, 0.7])]
}

# 질문마다 같은 후보를 사용해 최종 상위 3개를 평가합니다.
practice_rows = []
for item in practice_questions:
    expected_ids = set(item["relevant_doc_ids"])
    candidates = [practice_eval_bm25.invoke(item["question"]),
                  practice_eval_dense.invoke(item["question"])]

    # 같은 후보의 RRF 점수를 가중치별로 계산하고 내림차순으로 정렬합니다.
    method_results = {"BM25": candidates[0], "Dense": candidates[1]}
    for name, retriever in practice_hybrids.items():
        method_results[name] = retriever.weighted_reciprocal_rank(candidates)

    for method, candidate_documents in method_results.items():
        results = candidate_documents[:3]
        retrieved_ids = [doc.metadata["source_id"] for doc in results]
        practice_rows.append({
            "question_id": item["question_id"], "method": method,
            "expected_count": len(expected_ids),
            "found_count": len(expected_ids & set(retrieved_ids)),
            "recall_at_3": source_recall(results, expected_ids),
            "retrieved_ids": retrieved_ids,
            "missing_ids": sorted(expected_ids - set(retrieved_ids)),
            "characters": sum(len(doc.page_content) for doc in results),
        })

with pd.option_context("display.max_columns", None):
    display(pd.DataFrame(practice_rows))


In [ ]:
practice_frame = pd.DataFrame(practice_rows)
practice_scores = practice_frame.pivot(index="question_id", columns="method", values="recall_at_3")
practice_scores = practice_scores.reindex(
    index=[item["question_id"] for item in practice_questions],
    columns=["BM25", "Dense", *practice_hybrids],
)
display(practice_scores.round(3))
display(practice_frame.groupby("method", sort=False)[["recall_at_3"]].mean()
        .rename(columns={"recall_at_3": "mean_recall_at_3"}).round(3))


## 이번 강의 정리

| 개념 | 확인할 것 |
|---|---|
| BM25 | 같은 토큰화, 희귀도·빈도 포화·길이 보정, `k1`과 `k`의 구분 |
| Dense | 표현이 달라도 찾는가 |
| RRF | 1부터 세는 등수, `c`와 `k`의 구분, 가중치 순서 |
| Hybrid | 원문 ID로 중복 판단, 합친 뒤 최종 `k`로 자르기 |

## ⏭️ 다음 시간 예고

교안 02에서는 검색기를 그대로 두고 검색에 넣는 질문 자체를 바꾸는 질의 변환을 다룹니다.
